# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring a dataset with the [`mlcroissant`](https://github.com/mlcommons/croissant) library using the [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.github.io/croissant/), accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'
# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Accessing metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset DOI: {getattr(metadata, 'identifier', None)} \nLicense: {getattr(metadata, 'license', None)}\nVersion: {getattr(metadata, 'version', None)}")

## 2. Data Overview
Explore all available record sets in the dataset and their associated properties. We will list their `@id`s and show the available fields (by their `@id`) and columns for each record set.


> **Note:** All references are made via Croissant `@id`s for consistency.

In [ ]:
record_sets = list(dataset.record_sets)
print(f"Record sets found in the dataset ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    # List fields
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields: {field_ids}")
    # List columns by column @id
    col_ids = [c.id for c in rs.columns]
    if col_ids:
        print(f"  Columns: {col_ids}")
    print()

## 3. Data Extraction
We load records from each available record set into a pandas DataFrame. All access is performed via the `@id` for each record set.

You can select a record set by its `@id` for further analysis.

In [ ]:
# Compile all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id}, shape: {df.shape}")

# If there is at least one record set, show the fields/columns of the first as example
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"First record set '@id': {first_rs_id}")
    print("Available columns in the DataFrame:", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print('No record sets were found in the dataset.')

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate numeric field filtering, normalization, and grouping. Replace the field and group `@id`s below with those relevant from above.

If you know that a record set contains a numeric column (such as a regression output field), you can perform transformation and grouping.

**Example:** Filtering a numeric regression output field by threshold, normalizing, and grouping by a categorical variable. All references use the Croissant `@id`.

In [ ]:
# Select record set and fields to analyze. Replace these with actual @ids from above.
example_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(example_record_set_id, pd.DataFrame())

print(f"Working with record set: {example_record_set_id}")

# Identify numeric fields (@ids) in DataFrame columns
numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
print(f"Numeric fields available: {numeric_fields}")
if not numeric_fields:
    print('No numeric fields found for EDA. Please check the record set.')
else:
    # We'll analyze the first numeric field found
    numeric_field_id = numeric_fields[0]
    print(f"Analyzing numeric field: {numeric_field_id}")

    # Example: filter where values > threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records ({numeric_field_id} > mean [{threshold:.2f}]): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
    )
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (first object-type field that is not numeric)
    categorical_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    if categorical_fields:
        group_field_id = categorical_fields[0]
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print('No categorical group field found for grouping.')

## 5. Visualization
Let's plot the distribution of the selected numeric field and show a boxplot by group (if a groupable field is available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print('No numeric fields available to visualize.')
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of numeric field: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot for grouped variable
    if categorical_fields:
        plt.figure(figsize=(12,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id} (boxplot)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the dataset metadata and record sets described by a Croissant schema.
- Explored all record sets, fields, and columns using `@id`s.
- Loaded data from each record set as pandas DataFrames.
- Performed basic exploratory data analysis: filtering, normalization, and grouping on numeric fields.
- Visualized the distribution and grouped statistics.

You can further extend this notebook to focus on particular research questions or tailor analyses by selecting relevant field `@id`s, leveraging the semantic structure provided by the Croissant schema.

_FAIR^2 data exploration powered by `mlcroissant`!_